# Préparer un inventaire de pages avant une refonte

Ce notebook proposé par [Klymax, création de sites web](https://klymax.fr/) aide à relire un inventaire de pages avant une création ou une refonte de site. Il signale les champs vides et les doublons de titres dans un fichier CSV. Ces contrôles servent à préparer une revue humaine ; ils ne mesurent ni un classement Google ni la qualité globale d’un contenu.

Le fichier attendu contient quatre colonnes : `url`, `title`, `description` et `h1`. Vous pouvez préparer ce tableau à partir de votre plan de site ou d’un export de votre outil d’analyse. Les données de démonstration utilisent le domaine réservé `example.com` et ne décrivent pas un client.

## Utilisation

1. Faites une copie de ce notebook dans votre espace.
2. Remplacez le contenu de `csv_source` par votre export, en conservant les noms des colonnes.
3. Exécutez la cellule et relisez les points signalés avec le responsable des contenus.

Le code utilise uniquement la bibliothèque standard de Python. Il ne contacte aucun site et n’envoie aucune donnée à un service externe. Ne publiez pas votre copie si elle contient des informations internes.


In [1]:
import csv
import io
import json
from collections import Counter
from urllib.parse import urlsplit

csv_source = """url,title,description,h1
https://example.com/,Entreprise exemple,Présentation de l’activité,Notre entreprise
https://example.com/services,Nos services,,Nos services
https://example.com/contact,Nos services,Contactez notre équipe,
"""

pages = list(csv.DictReader(io.StringIO(csv_source)))
required = {"url", "title", "description", "h1"}
if not pages or not required.issubset(pages[0]):
    raise ValueError("Le CSV doit contenir url, title, description et h1.")

def clean(value):
    return " ".join((value or "").split())

titles = Counter(clean(row["title"]).casefold() for row in pages if clean(row["title"]))
review = []
for number, row in enumerate(pages, start=2):
    issues = []
    address = clean(row["url"])
    parsed = urlsplit(address)
    if parsed.scheme not in ("http", "https") or not parsed.netloc:
        issues.append("URL absolue à vérifier")
    for field in ("title", "description", "h1"):
        if not clean(row[field]):
            issues.append(field + " manquant")
    if clean(row["title"]) and titles[clean(row["title"]).casefold()] > 1:
        issues.append("Titre identique sur plusieurs pages")
    if issues:
        review.append({"ligne_csv": number, "url": address, "points_a_relire": issues})

print(json.dumps({"pages_lues": len(pages), "pages_a_relire": len(review), "resultats": review}, ensure_ascii=False, indent=2))


{
  "pages_lues": 3,
  "pages_a_relire": 2,
  "resultats": [
    {
      "ligne_csv": 3,
      "url": "https://example.com/services",
      "points_a_relire": [
        "description manquant",
        "Titre identique sur plusieurs pages"
      ]
    },
    {
      "ligne_csv": 4,
      "url": "https://example.com/contact",
      "points_a_relire": [
        "h1 manquant",
        "Titre identique sur plusieurs pages"
      ]
    }
  ]
}


## Interpréter les résultats

Un titre identique mérite une vérification : les pages peuvent avoir des objectifs différents et gagner à être distinguées. Un champ vide dans le CSV peut aussi venir d’un export incomplet. Ouvrez la page concernée avant de conclure qu’un élément manque sur le site.

Le script ne contrôle pas les redirections, les codes HTTP, les balises canoniques, les consignes d’indexation ou le rendu JavaScript. Il n’impose pas de longueur fixe aux titres et aux descriptions. Complétez cet inventaire par une lecture des pages sur ordinateur et téléphone, puis vérifiez les formulaires et les parcours essentiels.

## Passer de l’inventaire au projet

La liste obtenue sert à organiser une revue humaine. Pour examiner d’autres dimensions du site, un [audit de site](https://klymax.fr/audit) peut compléter ce travail ; le notebook seul ne couvre pas les contrôles décrits plus haut.

Après avoir clarifié le rôle de chaque page, préparez [une maquette](https://klymax.fr/maquette) pour organiser les contenus. Ce cadrage aide ensuite à définir le périmètre de la [création de votre site internet](https://klymax.fr/services/creation-site-internet).

Document préparé par Dan Zimmerman pour Klymax, le 9 octobre 2026.
